In [43]:
import os
import gc
import torch
import datetime
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
import re

In [44]:
def format_time(seconds):
    return str(datetime.timedelta(seconds=round(seconds, 2)))


def clean_text(text):
    return text.strip()


def is_noise(text):
    return len(text.strip()) < 3

In [45]:
def get_speaker_robust(start, end, tracks):
    votes = {}

    for t in tracks:
        if "start" not in t or "end" not in t:
            continue

        overlap_start = max(start, t["start"])
        overlap_end = min(end, t["end"])

        if overlap_end > overlap_start:
            dur = overlap_end - overlap_start
            spk = t.get("speaker", "UNKNOWN")
            votes[spk] = votes.get(spk, 0) + dur

    return max(votes, key=votes.get) if votes else "UNKNOWN"

In [46]:
TEACHER_CUES = [
    "bugün", "şimdi", "dikkat", "örnek", "şöyle",
    "anlatıyorum", "bakın", "önemli", "hatırlayın"
]


def detect_teacher_smart(tracks, segments_map):
    durations = defaultdict(float)
    texts_by_speaker = defaultdict(list)

    # 1) duration
    for t in tracks:
        spk = t.get("speaker", "UNKNOWN")
        durations[spk] += (t["end"] - t["start"])

    # 2) text grouping
    for seg_list in segments_map.values():
        for seg in seg_list:
            spk = get_speaker_robust(seg["start"], seg["end"], tracks)
            texts_by_speaker[spk].append(seg["text"])

    # 3) NLP cues
    cue_score = defaultdict(int)

    for spk, texts in texts_by_speaker.items():
        joined = " ".join(texts).lower()

        for cue in TEACHER_CUES:
            cue_score[spk] += len(re.findall(cue, joined))

    # 4) final score
    scores = {}

    for spk in durations.keys():
        scores[spk] = (
            0.5 * durations[spk]
            + 0.3 * cue_score.get(spk, 0)
            + 0.2 * len(texts_by_speaker.get(spk, []))
        )

    return max(scores, key=scores.get) if scores else "UNKNOWN"

In [47]:
def download_blob(i, blob_name):
    file_path = f"/tmp/temp_{i}.mp4"
    bucket.blob(blob_name).download_to_filename(file_path)
    return i, file_path

In [48]:
print("🔥 Running diarization...")

sample_file = "/tmp/sample.mp4"
bucket.blob(segments[0]).download_to_filename(sample_file)

diar = diar_pipeline(
    sample_file,
    min_speakers=1,
    max_speakers=2
)

tracks = safe_tracks(diar)

os.remove(sample_file)

🔥 Running diarization...


In [49]:
with ThreadPoolExecutor(max_workers=6) as ex:
    files = list(ex.map(lambda x: download_blob(*x), enumerate(segments)))

In [50]:
results_map = {}

for i, file_path in sorted(files):

    print(f"🎙 Processing {i+1}/{len(files)}")

    result = whisper_model.transcribe(
        file_path,
        language="tr",
        fp16=True,
        beam_size=1,
        best_of=1,
        condition_on_previous_text=False
    )

    results_map[i] = result["segments"]

    os.remove(file_path)

    gc.collect()
    torch.cuda.empty_cache()

🎙 Processing 1/12
🎙 Processing 2/12
🎙 Processing 3/12
🎙 Processing 4/12
🎙 Processing 5/12
🎙 Processing 6/12
🎙 Processing 7/12
🎙 Processing 8/12
🎙 Processing 9/12
🎙 Processing 10/12
🎙 Processing 11/12
🎙 Processing 12/12


In [51]:
teacher = detect_teacher_smart(tracks, results_map)

print("✅ SMART TEACHER:", teacher)

✅ SMART TEACHER: UNKNOWN


In [52]:
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write("=== FAST ENGINE v2 (SMART DETECTOR) ===\n\n")

    for i in sorted(results_map.keys()):

        for seg in results_map[i]:

            text = clean_text(seg["text"])

            if is_noise(text):
                continue

            speaker = get_speaker_robust(seg["start"], seg["end"], tracks)

            label = "HOCA" if speaker == teacher else "ÖĞRENCİ"

            start = format_time(seg["start"])
            end = format_time(seg["end"])

            line = f"[{start} → {end}] {label}: {text}\n"

            print(line.strip())
            f.write(line)

print("\n🔥 DONE FAST ENGINE v2 + SMART DETECTOR →", OUTPUT_TXT)

[0:00:00 → 0:00:29.980000] HOCA: Altyazı M.K.
[0:00:30 → 0:00:59.980000] HOCA: Altyazı M.K.
[0:01:00 → 0:01:08.300000] HOCA: Merhaba sesim geliyor mu?
[0:01:12.240000 → 0:01:12.960000] HOCA: Evet.
[0:01:13.880000 → 0:01:16.200000] HOCA: Evet kameranı açabilir misin?
[0:01:19.260000 → 0:01:19.900000] HOCA: Rica ederim.
[0:01:20.340000 → 0:01:26.840000] HOCA: Bir şey bir ses geliyor annemden bir daha şey olmasını isteyeceğim.
[0:01:26.840000 → 0:01:29.980000] HOCA: Tamam tamam beklerim.
[0:01:30 → 0:01:39.540000] HOCA: Merhaba
[0:01:39.540000 → 0:01:42] HOCA: Merhaba
[0:01:42 → 0:01:45.160000] HOCA: Merhaba
[0:01:45.160000 → 0:01:46.140000] HOCA: Merhaba Zeki mi ismin?
[0:01:47.360000 → 0:01:48.120000] HOCA: Yok şey
[0:01:48.120000 → 0:01:50.540000] HOCA: Burak da ismi babamın adı Zeki
[0:01:50.540000 → 0:01:52.060000] HOCA: Tamam
[0:01:52.060000 → 0:01:54.920000] HOCA: Burak
[0:01:54.920000 → 0:01:56.820000] HOCA: İsmini değiştireyim ben senin o zaman
[0:01:56.820000 → 0:01:59.080000] H